# Simulation Confidence Analysis

This notebook demonstrates the confidence analysis tools for quantifying
uncertainty in Monte Carlo simulation results. The key question: **how many
simulations do we need to be confident in the 95th percentile?**

Two approaches are available:
1. **Analytical** — uses exact portfolio moments (no simulation needed)
2. **Empirical** — uses order statistics from simulation output

The analytical approach accounts for heterogeneous mortality rates (individual
qx values) and volume concentration via the Cornish-Fisher expansion.

In [1]:
import numpy as np
import pandas as pd

from mortality_simulations import (
    stochastic_runs_hybrid,
    compute_portfolio_moments,
    estimate_quantile_ci_width,
    analyze_simulation_confidence,
    estimate_required_simulations,
    generate_confidence_summary,
    print_confidence_summary,
)

## 1. Create a sample portfolio

In [2]:
np.random.seed(42)
n_lives = 1_000

data = pd.DataFrame({
    "volume":      np.random.uniform(100_000, 50_000_000, n_lives),
    "baseline_qx": np.random.uniform(0.001, 0.05, n_lives),
    "shocked_qx":  np.random.uniform(0.002, 0.08, n_lives),
})

print(f"Portfolio: {n_lives:,} lives")
print(f"Volume range: {data['volume'].min():,.0f} - {data['volume'].max():,.0f}")
print(f"Shocked qx range: {data['shocked_qx'].min():.4f} - {data['shocked_qx'].max():.4f}")
data.describe()

Portfolio: 1,000 lives
Volume range: 331,138 - 49,985,912
Shocked qx range: 0.0020 - 0.0798


,volume,baseline_qx,shocked_qx
count,1.000000e+03,1000.000000,1000.000000
mean,2.456380e+07,0.025844,0.041188
std,1.457765e+07,0.014317,0.022673
min,3.311379e+05,0.001158,0.002001
25%,1.187507e+07,0.012813,0.022385
50%,2.489069e+07,0.026418,0.041048
75%,3.724155e+07,0.038263,0.061210
max,4.998591e+07,0.049971,0.079830


## 2. Analytical moments (no simulation needed)

Since each life is an independent Bernoulli trial, the exact moments of the
aggregate claim distribution can be computed directly:

- **Mean**: $\mu = \sum v_i \cdot q_{x_i}$
- **Variance**: $\sigma^2 = \sum v_i^2 \cdot q_{x_i}(1 - q_{x_i})$
- **Skewness**: $\gamma = \sum v_i^3 \cdot q_{x_i}(1-q_{x_i})(1-2q_{x_i}) \;/\; \sigma^3$

In [3]:
moments = compute_portfolio_moments(
    data["volume"].values,
    data["shocked_qx"].values,
)

for k, v in moments.items():
    if isinstance(v, float):
        print(f"  {k:>25}: {v:>24,.4f}")
    else:
        print(f"  {k:>25}: {v}")

                       mean:       1,016,518,698.9056
                   variance: 32,032,791,041,666,608.0000
                        std:         178,977,068.4799
                   skewness:                   0.1874
       third_central_moment: 1,074,631,304,439,427,256,811,520.0000
                    n_lives: 1000
                         cv:                   0.1761


## 3. Analytical CI width projection

Using the exact moments, we can project the 95% confidence interval width
for the 95th percentile at different simulation counts — without running
any simulations.

The standard error of the sample quantile is:

$$SE(\hat{\xi}_p) = \frac{\sqrt{p(1-p)/n} \cdot \sigma \cdot (1 + 2z_p\gamma/6)}{\phi(z_p)}$$

where the $(1 + 2z_p\gamma/6)$ term is the Cornish-Fisher density correction
for skewness.

In [4]:
print(f"{'Simulations':>15}  {'CI Width (abs)':>20}  {'CI Width (%)':>12}  {'SE':>18}")
print("-" * 72)

for n in [1_000, 10_000, 100_000, 1_000_000]:
    ci = estimate_quantile_ci_width(moments, n_simulations=n)
    print(
        f"{n:>15,}  "
        f"{ci['ci_width_absolute']:>20,.0f}  "
        f"{ci['ci_width_relative']:>11.3f}%  "
        f"{ci['se_quantile']:>18,.0f}"
    )

print()
ci = estimate_quantile_ci_width(moments, n_simulations=10_000)
print(f"Quantile estimate (normal):         {ci['quantile_estimate_normal']:>20,.0f}")
print(f"Quantile estimate (Cornish-Fisher):  {ci['quantile_estimate_cf']:>20,.0f}")
print(f"Method: {ci['method']}")

    Simulations        CI Width (abs)  CI Width (%)                  SE
------------------------------------------------------------------------
          1,000            51,701,019        3.915%          13,189,278
         10,000            16,349,298        1.238%           4,170,816
        100,000             5,170,102        0.392%           1,318,928
      1,000,000             1,634,930        0.124%             417,082

Quantile estimate (normal):                1,310,909,779
Quantile estimate (Cornish-Fisher):         1,320,445,999
Method: cornish_fisher


## 4. Run simulation and get empirical CI

Now run 10,000 simulations and compare the empirical order-statistics CI
against the analytical projection.

In [5]:
results = stochastic_runs_hybrid(
    data,
    n_trials=10_000,
    volume_col="volume",
    baseline_qx_col="baseline_qx",
    shocked_qx_col="shocked_qx",
)

ci = analyze_simulation_confidence(results, "claim_volume_shocked")
print(f"Empirical 95th percentile: {ci['point_estimate']:,.0f}")
print(f"95% CI: [{ci['ci_lower']:,.0f}, {ci['ci_upper']:,.0f}]")
print(f"CI width (relative): {ci['ci_width_relative']:.2f}%")

Empirical 95th percentile: 1,322,717,357
95% CI: [1,315,777,984, 1,329,936,564]
CI width (relative): 1.07%


## 5. Full summary: empirical vs analytical side-by-side

Passing `portfolio_data` to `generate_confidence_summary` enables the
analytical path alongside the empirical projections.

In [6]:
summary = generate_confidence_summary(
    results,
    "claim_volume_shocked",
    portfolio_data={
        "volumes": data["volume"].values,
        "qx": data["shocked_qx"].values,
    },
)

print_confidence_summary(summary)

SIMULATION CONFIDENCE ANALYSIS
Metric: claim_volume_shocked
Quantile: 95th percentile
Confidence Level: 95%

PORTFOLIO STRUCTURE (1,000 lives)
--------------------------------------------------
  Analytical mean:        1,016,518,698.91
  Analytical std dev:       178,977,068.48
  Coeff. of variation:             0.1761
  Skewness:                         0.1874

  Quantile estimates:
    Simulation (empirical):  1,322,717,357.40
    Normal approximation:    1,310,909,779.14
    Cornish-Fisher adjusted: 1,320,445,998.78

CURRENT RESULTS (10,000 simulations)
--------------------------------------------------
  Point estimate:         1,322,717,357.40
  CI lower bound:         1,315,777,983.80
  CI upper bound:         1,329,936,563.64
  CI width:                  14,158,579.84
  CI width (relative):               1.07%

PROJECTED CI WIDTH BY SIMULATION COUNT
----------------------------------------------------------------------
      Simulations    Empirical (%)   Analytical (%)  Status

## 6. Pilot run workflow

Run a cheap pilot (1,000 sims) and use `estimate_required_simulations`
to determine how many simulations are needed for a target precision.

In [7]:
pilot = stochastic_runs_hybrid(
    data,
    n_trials=1_000,
    volume_col="volume",
    baseline_qx_col="baseline_qx",
    shocked_qx_col="shocked_qx",
)

ci_pilot = analyze_simulation_confidence(pilot, "claim_volume_shocked")
print(f"Pilot (1,000 sims):")
print(f"  95th percentile: {ci_pilot['point_estimate']:,.0f}")
print(f"  CI width: {ci_pilot['ci_width_relative']:.2f}%")
print()

for target in [2.0, 1.0, 0.5]:
    est = estimate_required_simulations(
        pilot, "claim_volume_shocked",
        target_ci_width_relative=target,
    )
    print(f"  Target {target}% CI width -> {est['estimated_n_required']:>10,} simulations ({est['scaling_factor']:.1f}x)")

Pilot (1,000 sims):
  95th percentile: 1,319,024,369
  CI width: 3.89%

  Target 2.0% CI width ->      3,789 simulations (3.8x)
  Target 1.0% CI width ->     15,153 simulations (15.2x)
  Target 0.5% CI width ->     60,611 simulations (60.6x)


## 7. Volume concentration impact

A portfolio with a few very large policies ("whales") has higher CV and
skewness, requiring more simulations to achieve the same precision.
The analytical approach captures this directly.

In [8]:
np.random.seed(42)
n = 1_000

# Portfolio A: uniform volumes
vol_uniform = np.ones(n) * 5_000_000
qx_uniform = np.ones(n) * 0.02

# Portfolio B: concentrated volumes (10 whale policies)
vol_concentrated = np.random.exponential(2_000_000, n)
vol_concentrated[:10] = np.random.uniform(100_000_000, 500_000_000, 10)
qx_varied = np.random.uniform(0.001, 0.06, n)

moments_a = compute_portfolio_moments(vol_uniform, qx_uniform)
moments_b = compute_portfolio_moments(vol_concentrated, qx_varied)

print(f"{'':>30} {'Uniform':>18} {'Concentrated':>18}")
print("-" * 68)
print(f"{'Mean':>30} {moments_a['mean']:>18,.0f} {moments_b['mean']:>18,.0f}")
print(f"{'Std Dev':>30} {moments_a['std']:>18,.0f} {moments_b['std']:>18,.0f}")
print(f"{'CV':>30} {moments_a['cv']:>18.4f} {moments_b['cv']:>18.4f}")
print(f"{'Skewness':>30} {moments_a['skewness']:>18.4f} {moments_b['skewness']:>18.4f}")

print()
print("95th percentile CI width:")
print(f"{'Simulations':>15}  {'Uniform CI (%)':>15}  {'Concentrated CI (%)':>20}")
print("-" * 55)
for n_sim in [10_000, 100_000, 1_000_000]:
    ci_a = estimate_quantile_ci_width(moments_a, n_sim)
    ci_b = estimate_quantile_ci_width(moments_b, n_sim)
    print(
        f"{n_sim:>15,}  "
        f"{ci_a['ci_width_relative']:>14.3f}%  "
        f"{ci_b['ci_width_relative']:>19.3f}%"
    )

                                          Uniform       Concentrated
--------------------------------------------------------------------
                          Mean        100,000,000        178,480,615
                       Std Dev         22,135,944        205,400,314
                            CV             0.2214             1.1508
                      Skewness             0.2168             1.7167

95th percentile CI width:
    Simulations   Uniform CI (%)   Concentrated CI (%)
-------------------------------------------------------
         10,000           1.489%                5.357%
        100,000           0.471%                1.694%
      1,000,000           0.149%                0.536%


## 8. Baseline vs shocked comparison

Compare confidence for different metrics from the same simulation run.

In [9]:
metrics = [
    ("claim_volume_baseline", "baseline_qx"),
    ("claim_volume_shocked", "shocked_qx"),
]

for metric, qx_col in metrics:
    ci_emp = analyze_simulation_confidence(results, metric)
    m = compute_portfolio_moments(data["volume"].values, data[qx_col].values)
    ci_ana = estimate_quantile_ci_width(m, n_simulations=10_000)

    print(f"{metric}")
    print(f"  Empirical 95th pctile: {ci_emp['point_estimate']:>20,.0f}")
    print(f"  Analytical (CF):       {ci_ana['quantile_estimate_cf']:>20,.0f}")
    print(f"  Empirical CI width:    {ci_emp['ci_width_relative']:>19.2f}%")
    print(f"  Analytical CI width:   {ci_ana['ci_width_relative']:>19.2f}%")
    print(f"  Portfolio skewness:    {m['skewness']:>19.4f}")
    print()

claim_volume_baseline
  Empirical 95th pctile:          887,643,375
  Analytical (CF):                887,529,758
  Empirical CI width:                   1.64%
  Analytical CI width:                  1.52%
  Portfolio skewness:                 0.2445

claim_volume_shocked
  Empirical 95th pctile:        1,322,717,357
  Analytical (CF):              1,320,445,999
  Empirical CI width:                   1.07%
  Analytical CI width:                  1.24%
  Portfolio skewness:                 0.1874

